# Feature Extraction Test Notebook

This notebook tests TF-IDF feature extraction and resume-job matching using cosine similarity.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
src_path = project_root / 'src'
dataset_dir = project_root / 'Dataset'
train_csv = dataset_dir / 'train.csv'
test_csv = dataset_dir / 'test.csv'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from data_loader import load_train_test_data
from preprocessing import preprocess_dataframe
from feature_extraction import (
    create_tfidf_vectorizer,
    fit_tfidf,
    transform_tfidf,
    extract_train_test_features,
    rank_resumes_by_job_description,
)

print('Project root :', project_root)
print('Train exists :', train_csv.exists())
print('Test exists  :', test_csv.exists())

Project root : c:\Users\POOJA\OneDrive\Desktop\project
Train exists : True
Test exists  : True


## 1) Load and preprocess train/test data

In [ ]:
train_df, test_df = load_train_test_data(dataset_dir)

train_processed = preprocess_dataframe(train_df, text_column='text')
test_processed = preprocess_dataframe(test_df, text_column='text')

print('Train shape:', train_processed.shape)
print('Test shape :', test_processed.shape)
display(train_processed.head(2))

## 2) Test TF-IDF fit/transform

In [ ]:
vectorizer = create_tfidf_vectorizer(max_features=3000, ngram_range=(1, 2), min_df=1)
vectorizer, x_train = fit_tfidf(train_processed['text'], vectorizer)
x_test = transform_tfidf(vectorizer, test_processed['text'])

print('x_train shape:', x_train.shape)
print('x_test shape :', x_test.shape)
print('Vocabulary size:', len(vectorizer.vocabulary_))

## 3) Test one-call train/test feature extraction

In [ ]:
vec2, x_train2, x_test2 = extract_train_test_features(
    train_processed,
    test_processed,
    text_column='text',
    max_features=4000,
    ngram_range=(1, 2),
    min_df=1,
)

summary_df = pd.DataFrame({
    'Matrix': ['x_train2', 'x_test2'],
    'Rows': [x_train2.shape[0], x_test2.shape[0]],
    'Columns': [x_train2.shape[1], x_test2.shape[1]],
})
summary_df

## 4) Resume ranking against a job description

In [ ]:
sample_resumes = train_processed['text'].head(8).tolist()
job_description = 'Looking for data scientist with python machine learning and statistics skills.'

ranking_df = rank_resumes_by_job_description(job_description, sample_resumes, top_k=5)
ranking_df